In [1]:
import sys
import numpy as np
import anndata as ad
import polars as pl
import pickle
import src as scit

/home/aleksander-work/Skrivbord/pyproject/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import loompy

In [3]:
ds = loompy.connect('private/data/snRNAseq_nomitoribogonads.loom')
Xsp = ds.layer[''].sparse().tocsr()

In [4]:
Xsp.data

array([1., 1., 1., ..., 1., 1., 1.])

In [5]:
ks_ca = ds.ca.keys()
ks_ra = ds.ra.keys()

In [6]:
Xsp.data

array([1., 1., 1., ..., 1., 1., 1.])

In [28]:
rna = ad.AnnData(Xsp.T)

In [29]:
rna

AnnData object with n_obs × n_vars = 18493 × 15293

In [30]:
for k in ks_ca:
    rna.obs[k] = ds.ca[k]
for k in ks_ra:
    rna.var[k] = ds.ra[k]

In [31]:
ks_ca

['CellID',
 'G2M.Score',
 'Phase',
 'S.Score',
 'SCT_snn_res.0.5',
 'SCT_snn_res.0.6',
 'cell',
 'clusters_old',
 'cond',
 'nCount_RNA',
 'nCount_SCT',
 'nFeature_RNA',
 'nFeature_SCT',
 'old.ident',
 'orig.ident',
 'rep',
 'seurat_clusters']

In [33]:
rna.var_names = rna.var['Gene']
rna.var

,Gene
Gene,
l(2)gl,l(2)gl
CR11023,CR11023
spen,spen
kis,kis
Cda5,Cda5
...,...
CR45496,CR45496
CG42577,CG42577
CR34311,CR34311


In [34]:
anno = pl.read_csv('private/data/emb_nocc_nog_clusters.csv').to_dict()
anno = dict(zip(anno['name'], anno['value']))
anno

{'c_r1_CELL13_N2': 'Epidermis',
 'c_r1_CELL19_N5': 'Muscle somatic',
 'c_r1_CELL23_N2': 'Glia',
 'c_r1_CELL25_N2': 'Midgut',
 'c_r1_CELL41_N4': 'Epidermis',
 'c_r1_CELL44_N3': 'Muscle somatic',
 'c_r1_CELL49_N2': 'Epidermis',
 'c_r1_CELL55_N3': 'Epidermis',
 'c_r1_CELL59_N3': 'Muscle somatic',
 'c_r1_CELL60_N3': 'Neuronal 1',
 'c_r1_CELL63_N4': 'Midgut',
 'c_r1_CELL67_N3': 'Plasmatocytes',
 'c_r1_CELL80_N2': 'Midgut',
 'c_r1_CELL81_N3': 'Sense',
 'c_r1_CELL94_N2': 'Epidermis',
 'c_r1_CELL106_N3': 'Neuronal 2',
 'c_r1_CELL109_N2': 'Amnioserosa',
 'c_r1_CELL114_N5': 'Glia lateral',
 'c_r1_CELL125_N2': 'Yolk',
 'c_r1_CELL131_N4': 'Fat Body',
 'c_r1_CELL135_N3': 'Neuronal 1',
 'c_r1_CELL142_N3': 'Tracheal',
 'c_r1_CELL145_N2': 'Muscle visceral',
 'c_r1_CELL168_N5': 'Midgut',
 'c_r1_CELL177_N2': 'Muscle visceral',
 'c_r1_CELL178_N3': 'Plasmatocytes',
 'c_r1_CELL180_N2': 'Epidermis',
 'c_r1_CELL181_N2': 'Epidermis',
 'c_r1_CELL184_N3': 'Salivary Gland',
 'c_r1_CELL188_N2': 'Neuronal 2',
 'c_

In [35]:
rna.obs['label'] = rna.obs['CellID'].map(anno)
rna.obs

,CellID,G2M.Score,Phase,S.Score,SCT_snn_res.0.5,SCT_snn_res.0.6,cell,clusters_old,cond,nCount_RNA,nCount_SCT,nFeature_RNA,nFeature_SCT,old.ident,orig.ident,rep,seurat_clusters,label
0,c_r1_CELL13_N2,-0.004924,G1,-0.004652,4,1,c_r1_CELL13_N2,1,Control,412.0,486.0,186,182,1,1,c_r1,2,Epidermis
1,c_r1_CELL19_N5,-0.033890,G1,-0.013570,3,4,c_r1_CELL19_N5,4,Control,611.0,577.0,316,306,4,1,c_r1,3,Muscle somatic
2,c_r1_CELL23_N2,-0.039053,G1,-0.013570,9,12,c_r1_CELL23_N2,12,Control,800.0,627.0,365,343,12,1,c_r1,8,Glia
3,c_r1_CELL25_N2,0.007819,S,0.041137,6,5,c_r1_CELL25_N2,5,Control,503.0,502.0,288,277,5,1,c_r1,5,Midgut
4,c_r1_CELL41_N4,-0.030128,S,0.029388,4,1,c_r1_CELL41_N4,1,Control,526.0,509.0,301,286,1,1,c_r1,2,Epidermis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18488,kd_r2_CELL41467_N1,0.018736,G2M,-0.010234,13,16,kd_r2_CELL41467_N1,16,Mef2EzKD,463.0,483.0,258,241,16,2,kd_r2,12,Salivary Gland
18489,kd_r2_CELL41480_N1,-0.004236,G1,-0.008374,14,8,kd_r2_CELL41480_N1,8,Mef2EzKD,365.0,412.0,253,238,8,2,kd_r2,13,Epidermis head
18490,kd_r2_CELL41485_N1,-0.002440,G1,-0.009304,7,7,kd_r2_CELL41485_N1,7,Mef2EzKD,339.0,427.0,219,211,7,2,kd_r2,6,Fat Body
18491,kd_r2_CELL41564_N1,-0.027476,S,0.013046,4,1,kd_r2_CELL41564_N1,1,Mef2EzKD,267.0,403.0,239,228,1,2,kd_r2,2,Epidermis


In [36]:
rna.write_h5ad('private/data/snRNAseq-nogonad.h5ad')